# Silver — Customers & Sellers

Standardize text fields, deduplicate, and verify seller uniqueness.

In [ ]:
import sys

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.transformations.silver_entities import (
    SilverEntitiesConfig, build_silver_customers, build_silver_sellers,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = SilverEntitiesConfig()

In [ ]:
customers, cust_stats = build_silver_customers(spark, config)
customers.write.format("delta").mode("overwrite").saveAsTable(config.silver_customers)
print("Customers:", cust_stats)

In [ ]:
sellers, seller_stats = build_silver_sellers(spark, config)
sellers.write.format("delta").mode("overwrite").saveAsTable(config.silver_sellers)
print("Sellers:", seller_stats)

In [ ]:
import json

report = {
    "silver_customers": config.silver_customers,
    "silver_sellers": config.silver_sellers,
    "customer_stats": cust_stats,
    "seller_stats": seller_stats,
}
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(report, dbutils, "/Volumes/globalmart/metadata/run_reports/silver_entities_latest.json")
except Exception as exc:
    print("Volume save skipped:", exc)